In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import copy
import pandas as pd
import time

In [2]:
#from utils.subspace_clustering_helper_funcs import *
from preprocessing import *

## Loading in the data

In [3]:
brc_path = "C:\\Users\\kdmen\\Box\\Yamagami Lab\\Data\\2024_UIST_dataset\\upload\\segmented_filtered_data"

In [ ]:
# pID 101 doesn't exist
# each participant has 100 experimenter defined files and 50 user defined files
# 10 experimenter defined gestures and 5 user defined gestures

file_types = ["IMU_extract", "movavg_files"]
expt_types = ["standardized"] # user-defined, experimenter-defined, rehab

## Rehab is variations (of personalized / 'experimenter-defined')!
pIDs_impaired = ['P102','P103','P104','P105','P106','P107','P108','P109','P110','P111',
       'P112','P114','P115','P116','P118','P119','P121','P122','P123','P124','P125',
       'P126','P127','P128', 'P131', 'P132']
# remove participants P001 and P003 because they dont have duplicate or open gestures
pIDs_unimpaired = ['P004','P005','P006','P008','P010','P011']

pIDs_both = pIDs_impaired + pIDs_unimpaired

Version using dataframes (wasted way too much time writing this...)

In [5]:
import numpy as np
from scipy.interpolate import interp1d

def interpolate_df(df, num_rows=64, columns_to_exclude=None):
    if columns_to_exclude is None:
        columns_to_exclude = []
    
    # Identify numeric columns to interpolate
    cols_to_interpolate = [c for c in df.columns if c not in columns_to_exclude]
    
    # Create the old index (current length) and new index (target length)
    old_indices = np.linspace(0, 1, len(df))
    new_indices = np.linspace(0, 1, num_rows)
    
    new_data = {}
    for col in cols_to_interpolate:
        # Linear interpolation for each sensor channel
        f = interp1d(old_indices, df[col].values, kind='linear')
        new_data[col] = f(new_indices)
    
    # Re-add excluded columns (taking the first value since they are metadata)
    for col in columns_to_exclude:
        if col in df.columns:
            new_data[col] = [df[col].iloc[0]] * num_rows
            
    return pd.DataFrame(new_data)

In [6]:
import os
import pandas as pd

def load_data(pIDs, data_dir_path="C:\\Users\\kdmen\\Box\\Yamagami Lab\\Data\\2024_UIST_dataset\\upload\\segmented_filtered_data\\", 
              file_types=["IMU_extract", "movavg_files"], expt_types=["standardized"]):
    
    # Define headers outside the loop to save overhead
    emg_headers = [f'EMG{i}' for i in range(1, 17)]
    imu_headers = [
        'IMU1_ax', 'IMU1_ay', 'IMU1_az', 'IMU1_vx', 'IMU1_vy', 'IMU1_vz',
        'IMU2_ax', 'IMU2_ay', 'IMU2_az', 'IMU2_vx', 'IMU2_vy', 'IMU2_vz',
        'IMU3_ax', 'IMU3_ay', 'IMU3_az', 'IMU3_vx', 'IMU3_vy', 'IMU3_vz',
        'IMU4_ax', 'IMU4_ay', 'IMU4_az', 'IMU4_vx', 'IMU4_vy', 'IMU4_vz',
        'IMU5_ax', 'IMU5_ay', 'IMU5_az', 'IMU5_vx', 'IMU5_vy', 'IMU5_vz',
        'IMU6_ax', 'IMU6_ay', 'IMU6_az', 'IMU6_vx', 'IMU6_vy', 'IMU6_vz',
        'IMU7_ax', 'IMU7_ay', 'IMU7_az', 'IMU7_vx', 'IMU7_vy', 'IMU7_vz',
        'IMU8_ax', 'IMU8_ay', 'IMU8_az', 'IMU8_vx', 'IMU8_vy', 'IMU8_vz',
        'IMU9_ax', 'IMU9_ay', 'IMU9_az', 'IMU9_vx', 'IMU9_vy', 'IMU9_vz',
        'IMU11_ax', 'IMU11_ay', 'IMU11_az', 'IMU11_vx', 'IMU11_vy', 'IMU11_vz',
        'IMU13_ax', 'IMU13_ay', 'IMU13_az', 'IMU13_vx', 'IMU13_vy', 'IMU13_vz',
        'IMU15_ax', 'IMU15_ay', 'IMU15_az', 'IMU15_vx', 'IMU15_vy', 'IMU15_vz'
    ]

    data_dict = {}
    
    for expt_type in expt_types:
        for pid in pIDs:
            print(f"Processing Participant: {pid}")
            pid_path = os.path.join(data_dir_path, pid)
            
            for file_type in file_types:
                sub_path = os.path.join(pid_path, file_type)
                
                if not os.path.exists(sub_path):
                    print(f"Subpath does not exist: {sub_path}")
                    continue
                
                # --- NEW: Check if folder is empty ---
                files_in_folder = os.listdir(sub_path)
                if not files_in_folder:
                    print(f"Folder {file_type} is empty!")
                    continue
                
                for file in files_in_folder:
                    if expt_type not in file:
                        continue 
                         
                    split_filename = file.split("_")
                    if len(split_filename) < 6:
                        print(f"Unexpected filename format: {file}")
                        continue
                        
                    gestureID = split_filename[4]
                    gestureNum = split_filename[5]
                    
                    headers = emg_headers if file_type == "movavg_files" else imu_headers

                    file_path = os.path.join(sub_path, file)
                    # Use header=0 if your CSVs have a header row, header=None if they don't
                    df = pd.read_csv(file_path, names=headers, header=0)
                    
                    if df.empty:
                        print(f"DataFrame is empty for file: {file_path}")
                        continue

                    # Metadata columns
                    df['Participant'] = pid
                    df['Gesture_ID'] = gestureID
                    df['Gesture_Num'] = gestureNum

                    # Interpolate to consistent 64-row length
                    df_interpolated = interpolate_df(df, num_rows=64, 
                                                     columns_to_exclude=['Participant', 'Gesture_ID', 'Gesture_Num'])

                    unique_key = (pid, gestureID, gestureNum)

                    if unique_key in data_dict:
                        # Join new columns only (avoiding duplicate Participant/ID columns)
                        existing_df = data_dict[unique_key]
                        new_cols = df_interpolated.drop(columns=['Participant', 'Gesture_ID', 'Gesture_Num'])
                        data_dict[unique_key] = pd.concat([existing_df, new_cols], axis=1)
                    else:
                        data_dict[unique_key] = df_interpolated

    if not data_dict:
        print("No data was loaded. Check your paths and filters.")
        return pd.DataFrame()

    data_lst = list(data_dict.values())
    
    # Validate shapes against the target (64 rows, 91 columns is your expectation)
    expected_rows = 64
    # Filter out anything that didn't merge correctly
    edited_data_lst = [ele_df for ele_df in data_lst if ele_df.shape[0] == expected_rows]
    
    if not edited_data_lst:
        print("None of the loaded DataFrames matched the expected shape.")
        return pd.DataFrame()

    dataframe = pd.concat(edited_data_lst, ignore_index=True)

    print(f"Final Dataframe Shape: {dataframe.shape}")
    nan_count = dataframe['Participant'].isna().sum()
    print(f"Number of rows with NaN Participant: {nan_count}")
        
    return dataframe

# EMG and IMU Dataset

In [7]:
start_time = time.time()
data_df = load_data(pIDs_both, data_dir_path=brc_path)
end_time = time.time()

print(f"\nCompleted in {end_time - start_time}s")

Processing Participant: P102
Folder IMU_extract is empty!
Folder movavg_files is empty!
Processing Participant: P103
Folder IMU_extract is empty!
Folder movavg_files is empty!
Processing Participant: P104
Processing Participant: P105
Folder IMU_extract is empty!
Folder movavg_files is empty!
Processing Participant: P106
Processing Participant: P107
Folder IMU_extract is empty!
Folder movavg_files is empty!
Processing Participant: P108
Folder IMU_extract is empty!
Folder movavg_files is empty!
Processing Participant: P109
Folder IMU_extract is empty!
Folder movavg_files is empty!
Processing Participant: P110
Folder IMU_extract is empty!
Folder movavg_files is empty!
Processing Participant: P111
Folder IMU_extract is empty!
Folder movavg_files is empty!
Processing Participant: P112
Folder IMU_extract is empty!
Folder movavg_files is empty!
Processing Participant: P114
Folder IMU_extract is empty!
Folder movavg_files is empty!
Processing Participant: P115
Processing Participant: P116
Fold

In [8]:
print(data_df.shape)
data_df.head()

(10560, 91)


,IMU1_ax,IMU1_ay,IMU1_az,IMU1_vx,IMU1_vy,IMU1_vz,IMU2_ax,IMU2_ay,IMU2_az,IMU2_vx,...,EMG7,EMG8,EMG9,EMG10,EMG11,EMG12,EMG13,EMG14,EMG15,EMG16
0,0.199707,-0.876953,0.399902,-0.032991,-0.190496,0.014899,-0.340332,-0.837891,-0.505859,0.145799,...,8.559906e-07,0.000005,0.000001,0.000001,0.000001,0.000002,0.000001,0.000001,0.000001,0.000002
1,0.205566,-0.874202,0.403320,-0.037417,-0.196544,0.018430,-0.340022,-0.839433,-0.510665,0.147589,...,1.046315e-06,0.000004,0.000001,0.000001,0.000001,0.000002,0.000001,0.000001,0.000001,0.000002
2,0.204210,-0.871977,0.405273,-0.044697,-0.196105,0.015575,-0.337092,-0.838115,-0.511851,0.144346,...,9.750373e-07,0.000006,0.000001,0.000001,0.000001,0.000002,0.000001,0.000001,0.000001,0.000002
3,0.206241,-0.876767,0.404064,-0.063195,-0.235143,0.022754,-0.342053,-0.841564,-0.509742,0.159026,...,1.007400e-06,0.000007,0.000001,0.000001,0.000001,0.000002,0.000001,0.000002,0.000002,0.000002
4,0.201350,-0.875620,0.403274,-0.042856,-0.195142,0.020220,-0.340107,-0.841309,-0.508347,0.146289,...,1.777024e-06,0.000011,0.000001,0.000001,0.000001,0.000002,0.000002,0.000001,0.000002,0.000002


In [9]:
# Example dataframe, assuming 'df' is your dataframe
# Count NaNs per row
nans_per_row = data_df.isna().sum(axis=1)

# Count NaNs per column
nans_per_column = data_df.isna().sum(axis=0)


In [10]:
# Summary statistics for NaNs per row
nans_per_row.describe()

# Summary statistics for NaNs per column
#nans_per_column.describe()


count    10560.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
dtype: float64

In [11]:
assert(False)

AssertionError: 

# EMG Only Dataset

In [ ]:
#start_time = time.time()
#emg_df = load_data(pIDs_both, file_types=["movavg_files"], data_dir_path=brc_path)
#end_time = time.time()

#print(f"\nCompleted in {end_time - start_time}s")

# QUICKER TO JUST SLICE THE ALREADY LOADED IN DF FROM ABOVE!
# Select columns that contain "emg" in their name
emg_columns = [col for col in data_df.columns if 'emg' in col]
emg_df = data_df[emg_columns]

In [ ]:
print(emg_df.shape)
emg_df.head()

In [ ]:
# Example dataframe, assuming 'df' is your dataframe
# Count NaNs per row
nans_per_row = emg_df.isna().sum(axis=1)

# Count NaNs per column
nans_per_column = emg_df.isna().sum(axis=0)


In [ ]:
# Summary statistics for NaNs per row
nans_per_row.describe()

# Summary statistics for NaNs per column
#nans_per_column.describe()


# IMU Only Dataset

In [ ]:
#start_time = time.time()
#imu_df = load_data(pIDs_both, file_types=["IMU_extract"], data_dir_path=brc_path)
#end_time = time.time()
#
#print(f"\nCompleted in {end_time - start_time}s")

# QUICKER TO JUST SLICE THE ALREADY LOADED IN DF FROM ABOVE!
# Select columns that contain "emg" in their name
imu_columns = [col for col in data_df.columns if 'imu' in col]
imu_df = data_df[imu_columns]

In [ ]:
print(imu_df.shape)
imu_df.head()

In [ ]:
# Example dataframe, assuming 'df' is your dataframe
# Count NaNs per row
nans_per_row = imu_df.isna().sum(axis=1)

# Count NaNs per column
nans_per_column = imu_df.isna().sum(axis=0)


In [ ]:
# Summary statistics for NaNs per row
nans_per_row.describe()

# Summary statistics for NaNs per column
#nans_per_column.describe()


# Save the dataframe

In [ ]:
assert(False)

In [ ]:
brc_data_save_path = 'D:\\Kai_MetaGestureClustering_24\\saved_datasets\\filtered_datasets\\'

In [ ]:
## Pickle is theoretically faster for Python...

data_df.to_pickle(brc_data_save_path+'metadata_IMU_EMG_allgestures_allusers.pkl')
#df = pd.read_pickle('your_dataframe.pkl')

In [ ]:
emg_df.to_pickle(brc_data_save_path+'metadata_EMG_allgestures_allusers.pkl')

In [ ]:
imu_df.to_pickle(brc_data_save_path+'metadata_IMU_allgestures_allusers.pkl')